# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Record sets, fields, and columns are referenced by their `@id` fields. This allows for unique and clear referencing throughout.

In [ ]:
# List all record sets and their fields/columns by @id

record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for idx, rec in enumerate(record_sets):
    print(f"[{idx}] Record Set @id: {rec.id_}")
    # List fields
    print(f"    Fields in {rec.id_}:")
    for field in rec.fields:
        print(f"        Field @id: {field.id_}  (name: {field.name})")
        # List columns for each field (if any)
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"            Column @id: {col.id_}  (name: {col.name})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we select the first record set and list all its records, using their field `@id`s.

In [ ]:
# Extract data from each record set
# We'll use the @id of the first record set for demonstration

record_sets = dataset.metadata.record_sets
dataframes = {}
record_set_ids = [rec.id_ for rec in record_sets]

# Load all available record sets as DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the columns from the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We use field `@id`s (not names) to select and analyze fields.


In [ ]:
# Example EDA: Filter and normalize values of a numeric field
import numpy as np

# Choose a numeric field. You can adapt this by inspecting the fields from the overview above.
# For demonstration, we try to pick a likely numeric field by name, else ask user to adapt.
df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()

numeric_candidates = [col for col in df.columns if col.lower() in ('coverage_percent','mw','peptide_count','mw (da)','molecular_weight','abundance','peptide_counts','coverage','mw')] if not df.empty else []
if not numeric_candidates and not df.empty:
    # fallback: guess the first numeric column (by dtype)
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidates.append(col)
    # else take the first column
    if not numeric_candidates:
        numeric_candidates.append(df.columns[0]) if len(df.columns)>0 else None

if not df.empty and numeric_candidates:
    numeric_field = numeric_candidates[0]
    # Remove nulls for numeric calculation
    valid_df = df[df[numeric_field].apply(lambda x: isinstance(x, (int, float, np.integer, np.floating)) or (str(x).replace('.','',1).isdigit() if isinstance(x, str) else False))].copy()
    # Attempt to coerce to float
    valid_df[numeric_field] = valid_df[numeric_field].astype(float)
    
    threshold = valid_df[numeric_field].mean() if len(valid_df)>0 else 0
    filtered_df = valid_df[valid_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely group/categorical field
    group_candidates = [col for col in df.columns if col.lower() in ('protein_id','accession','description','group','category','sample')] if not df.empty else []
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No categorical/group field found to group by.")
else:
    print("No numeric field found or data is empty. Please check earlier steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and, if possible, compare means by a categorical field. You can adjust fields as needed based on the earlier overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if not df.empty and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_candidates:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric/categorical field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and perform basic analysis on the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` library. Using unique entity `@id`s ensured unambiguous referencing throughout.

You saw how to:
- Load the dataset schema and data
- Explore record sets, fields, and columns using their `@id`
- Load tabular records for analysis
- Apply data filtering and normalization
- Visualize field distributions

You can extend this notebook by exploring other record sets, fields, and more complex analyses specific to your scientific questions.